In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pyarrow
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid')

plt.rcParams['figure.figsize'] = (10, 6)

In [29]:
#Desactivar notacion científica
pd.set_option('display.float_format', '{:.2f}'.format)

### Importacion datos

In [2]:
import sqlite3
conn = sqlite3.connect("../data/raw/ecommerce.db")

In [3]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,2019-Oct
1,2019-Nov
2,2019-Dec
3,2020-Jan
4,2020-Feb


In [4]:
pd.read_sql('PRAGMA table_info("2019-Oct");', conn)

,cid,name,type,notnull,dflt_value,pk
0,0,index,BIGINT,0,None,0
1,1,event_time,TEXT,0,None,0
2,2,event_type,TEXT,0,None,0
3,3,product_id,BIGINT,0,None,0
4,4,category_id,BIGINT,0,None,0
5,5,category_code,TEXT,0,None,0
6,6,brand,TEXT,0,None,0
7,7,price,FLOAT,0,None,0
8,8,user_id,BIGINT,0,None,0
9,9,user_session,TEXT,0,None,0


In [6]:
oct19 = pd.read_sql('SELECT * FROM "2019-Oct";', conn)
nov19 = pd.read_sql('SELECT * FROM "2019-Nov";', conn)
dec19 = pd.read_sql('SELECT * FROM "2019-Dec";', conn)
jan20 = pd.read_sql('SELECT * FROM "2020-Jan";', conn)
feb20 = pd.read_sql('SELECT * FROM "2020-Feb";', conn)


In [7]:
print(oct19.shape, nov19.shape, dec19.shape, jan20.shape, feb20.shape)

(407925, 10) (462833, 10) (351304, 10) (443224, 10) (429790, 10)


In [8]:
print(oct19.columns, nov19.columns, dec19.columns, jan20.columns, feb20.columns)

Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str') Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str') Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str') Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str') Index(['index', 'event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='str')


Vamos a integrar los datos antes de hacer la calidad de datos. Solo en este caso especifico y por la naturaleza de las tablas, ya son muy parecidas y comparten excatamente las mismas columnas.

In [9]:
df = pd.concat([oct19, nov19, dec19, jan20, feb20], ignore_index=True, axis=0)

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2095076 entries, 0 to 2095075
Data columns (total 9 columns):
 #   Column         Dtype              
---  ------         -----              
 0   date           datetime64[us, UTC]
 1   event          str                
 2   product_id     int64              
 3   category       int64              
 4   category_code  str                
 5   brand          str                
 6   price          float64            
 7   user_id        int64              
 8   user_session   str                
dtypes: datetime64[us, UTC](1), float64(1), int64(3), str(4)
memory usage: 237.8 MB


In [21]:
df.head(1)

,date,event,product_id,category,category_code,brand,price,user_id,user_session
0,2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361


In [12]:
#cambio de tipos de variables
df.event_time = pd.to_datetime(df.event_time)

In [ ]:
#eliminar columna inutil
df = df.drop(columns=['index'])

In [19]:
#renombrar columnas
df.columns = ['date','event','product_id','category','category_code','brand','price','user_id','user_session']

#### Análisis de nulos

In [23]:
# ¿Existe alguna marca no nula para cada product_id?
has_brand = (
    df.groupby("product_id")["brand"]
      .transform(lambda s: s.notna().any())
)

# Registros cuyo brand es nulo pero cuyo producto sí tiene marca en otro registro
mask = df["brand"].isna() & has_brand

mask.sum()

np.int64(15255)

#### Decidimos eliminar las siguientes variables/registros:
1. category_code: la proporcion de nulos es muy elevada y no aporta informacion relevante ya que existe la variable categoria.
2. brand: la proporcion de nulos es elevada (casi la mitad de los registros) y la cantidad de registros potenciales con brand no nula es de 15255 solamente, asi que la cantidad total de nulos sigue siendo demasaido elevada. Como no disponemos de un diccionario de product_id - brand y el objetivo de éste análisis es puramente académico, eliminaremos la variable.
3. user_session: se eliminaran los registros con user_session nulo debido a que en proporcion son muy pocos registros respecto al total (506 de 2095076) y no afecta su ausencia para el objetivo de éste análisis académico. 

In [ ]:
#Eliminar columnas 
df = df.drop(columns=['category_code', 'brand'])

#Eliminar registros de user_session nulos
df = df.dropna(subset=['user_session'])

In [27]:
df.isna().sum()

date            0
event           0
product_id      0
category        0
price           0
user_id         0
user_session    0
dtype: int64

#### Análisis de variables numéricas

In [31]:
df.describe()

,product_id,category,price,user_id
count,2094570.00,2094570.00,2094570.00,2094570.00
mean,5487103.56,1553112489392098048.00,8.42,521077545.56
std,1300923.90,167907497920480608.00,19.14,87553855.76
min,3752.00,1487580004807082752.00,-47.62,4661182.00
25%,5724652.00,1487580005754995456.00,2.05,480613387.00
50%,5811665.00,1487580008246412288.00,4.00,553341613.00
75%,5858353.00,1487580013489291520.00,6.86,578406571.00
max,5932595.00,2242903426784559104.00,327.78,622087993.00


In [ ]:
#Comprovamos cantidad de precios "erroneos"
df[df['price'] <= 0].count().unique()

array([20544])

Debido a que la cantidad de precios erroneos es muy baja respecto al total de datos (0,98%), eliminaremos estos registros para que no interfieran con el cálculo de métricas posterior.

In [40]:
#Nos quedamos solo con registros con precio positivo
df = df[df['price'] > 0]

#### Analisis de variables categoricas

In [43]:
df.event.value_counts()

event
view                961558
cart                574547
remove_from_cart    410357
purchase            127564
Name: count, dtype: int64

In [44]:
df.category.value_counts()

category
1487580007675986893    109285
1487580005595612013     78777
1487580005092295511     76567
1487580005671109489     69751
1602943681873052386     63924
                        ...  
1487580013363462335         1
1487580008204469224         1
1487580011559911545         1
1487580009857024046         1
2053031020655018687         1
Name: count, Length: 508, dtype: int64

In [ ]:
df.to_parquet("../data/intermediate/df_clean.parquet", index=True)